# 1. Сбор данных

Мы решили изучать на этапе EDA "самые ликвидные" 10 компаний с 2010-2025. Чтобы не создать bias в данных, возьмём 10 компаний с самой высокой оценкой ежедневного денежного оборота (dollar volume) в среднем с 1 по 31 декабря 2009. Финансовые данные соберём с yfinance.

## 1.1. Выбор компаний

Сперва нужно понять, среди каких компаний можно найти те, что имеют самую высокую капитализацию. Что очень приятно, на Википедии есть статья со всеми компаниями в S&P500 с версией на 31 декабря 2009: https://en.wikipedia.org/w/index.php?title=List_of_S%26P_500_companies&oldid=335120317. Нам лишь остается достать оттуда список тикеров и найти топ 10 по dollar volume.

In [1]:
import io

import pandas as pd
import requests

WIKI_OLDID = 335120317 # версия на 31.12.2009
UA = {"User-Agent": "Nikita Kliuchnikov niklyuchnikov@edu.hse.ru"} # меняйте на свой!

html = requests.get("https://en.wikipedia.org/w/index.php",
                    params={
                        "title": "List_of_S&P_500_companies",
                        "oldid": WIKI_OLDID
                    },
                    headers=UA).text
wiki = pd.read_html(io.StringIO(html))[0]
sym_col = next(c for c in wiki.columns if "symbol" in str(c).lower() or "ticker" in str(c).lower())

# заменяем тикеры на формат yfinance:
tickers = sorted(wiki[sym_col].astype(str).str.strip().str.replace(".", "-", regex=False).unique().dropna())
print(len(tickers), "компаний:", tickers[:5], "и так далее")

500 компаний: ['A', 'AA', 'AAPL', 'ABC', 'ABT'] и так далее


Все версии страницы можно посмотреть здесь: https://en.wikipedia.org/w/index.php?title=List_of_S%26P_500_companies&action=history&dir=prev&limit=500.

Идея: в дальнейшем брать списки акций оттуда!

Теперь посчитаем dollar volume: берём цены закрытия (adjusted) и объём торгов (тоже adjusted). Утверждается, что если взять оба adjusted, то возможные корректировки правильно взаимо-вычтутся, чтобы правильно считалась оценка dollar volume. TODO: убедиться в справедливости этого.

Важный нюанс: это лишь оценка – настоящий dollar volume считался бы по цене каждой сделки!

In [2]:
import yfinance as yf
from IPython.display import clear_output

# здесь auto_adjust=True
prices_2009 = yf.download(tickers, start="2009-12-01", end="2010-01-01")
assert prices_2009 is not None, "Загрузка не удалась"
clear_output()

dollar_volume = prices_2009['Close'] * prices_2009['Volume']
adv = dollar_volume.mean().dropna()
top_tickers = adv.sort_values(ascending=False).head(10).index.tolist()
top_tickers

['AAPL', 'BAC', 'C', 'AMZN', 'GOOG', 'XOM', 'WFC', 'GS', 'MSFT', 'JPM']

## 1.2. Загрузка OHLCV данных

Для сборки свечей выбранных компаний используем снова yfinance.

Важно: это сырые данные без корректировок. Хочется сперва посмотреть на них.

In [3]:
ohlcv_top_not_adj = yf.download(top_tickers, start='2010-01-01', end='2026-01-01', auto_adjust=False)
assert ohlcv_top_not_adj is not None, "Загрузка не удалась"

[*********************100%***********************]  10 of 10 completed


Посмотрим на график сырой цены:

In [4]:
import plotly.express as px

assert ohlcv_top_not_adj is not None
close_raw = ohlcv_top_not_adj.Close[top_tickers] # в порядке убывания по ликвидности

fig = px.line(close_raw, title="Динамика Цен, Top-Market-Cap Акции на начало 2010 года")
fig.update_layout(xaxis={'title': {'text': "Дата"}}, yaxis={'title': {'text': "Цена закрытия"}})
fig.show()

Отнормируем так, чтобы все цены начинались в 1 (получаются кумулятивные доходности), а также возьмём логарифмическую шкалу.

In [5]:
fig = px.line(close_raw / close_raw.iloc[0],
              title="Динамика Цен, Top-Market-Cap Акции на начало 2010 года, Нормировка по начальной цене",
              log_y=True)
fig.update_layout(
    xaxis={'title': {
        'text': "Дата"
    }},
    yaxis={
        'title': {
            'text': "Кумулятивная доходность"
        },
        "showexponent": 'all', # TODO: пофиксить шкалу по y - это не помогло
        "exponentformat": 'e'
    },
)
fig.show()

Можем посмотреть на свечи (candlestick chart) для каждой акции. Вот, например, последние 100 наблюдений по Apple в датасете:

In [6]:
import plotly.graph_objects as go

assert ohlcv_top_not_adj is not None
aapl_raw_sample = ohlcv_top_not_adj[:, "AAPL"].iloc[-100:]

fig = go.Figure(data=[
    go.Candlestick(
        x=aapl_raw_sample.index,
        open=aapl_raw_sample.Open,
        high=aapl_raw_sample.High,
        low=aapl_raw_sample.Low,
        close=aapl_raw_sample.Close,
    )
])
fig.update_layout(
    title="OHLCV для AAPL – последние 100 наблюдений",
    xaxis_rangeslider_visible=False,
    xaxis={'title': {
        'text': "Дата"
    }},
    yaxis={'title': {
        'text': "Цена"
    }},
)
fig.show()